# Notebook 02: Link analysis and alert levels (Section 8, SOP 1)

Training wet seasons 2018-2022 for fitting, 2023 for checking; 2024 pumping data are not read. Operational rules: `configs/link.yaml` (fixed before the statistics were computed; gate-'O' and pump checks amended post hoc, decision P2-a). Gates G4, G20, G7, G8. Script: `python scripts/run_phase.py --phase 2`.

**Runs locally:** the raw data are committed in `data/raw/`; the first cell checks them against the manifest and, in a fresh clone, rebuilds any missing generated inputs (no Google Drive needed).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from pasig.local import prepare
prepare(2)   # raw data from the repo (no Drive); rebuilds missing local inputs once per clone
import pandas as pd
from pasig import phase2
from pasig.gates import require_gates, read_gate
require_gates('G6')

## Run all steps (writes gates, tables, figure and `configs/alert_levels.yaml`)

In [ ]:
out = phase2.run()

## 8.1 Status legend (G4) and Quiapo 2021 identity (G20)

In [ ]:
pd.read_csv('reports/tables/g4_legend_checks.csv')

In [ ]:
read_gate('G4')['decision'], read_gate('G20')['decision'], out['g20']['conditions']

## 8.2 First assumption: Pandacan vs Aviles river side

In [ ]:
out['first']

## 8.3 Grid tables

In [ ]:
out['grid'][['level_m','hours','stress_share','stress_ci_low','stress_ci_high','open_gate_share','dry_share','events','event_stress_share','event_stress_ci_low','event_stress_ci_high']].round(3)

![stress by level](../reports/figures/fig_link_stress_by_level.png)

## 8.4 Logistic regression and H1 (G7)

In [ ]:
lm = out['logit']
pd.DataFrame({k: lm[k] for k in ['or_L_per_0.1m_M1','or_L_per_0.1m_M2_at_mean_rain','interaction_M2','or_lnR_M1']}).T.round(4)

In [ ]:
lm['lr_M0_vs_M1_unclustered'], lm['lr_M1_vs_M2_unclustered'], out['auc'], out['h1']

### Quiapo secondary check (with and without 2021, D18) and Quiapo 2023 (descriptive only, decision (a))

In [ ]:
q = out['quiapo']
{k: q[k]['or_L_per_0.1m_M1'] for k in ('without_2021','with_2021') if k in q}, q.get('quiapo_2023_descriptive')

## 8.5 Alert levels (G8) and sensitivity

In [ ]:
print(open('configs/alert_levels.yaml').read())
out['sensitivity']

## Tests

In [ ]:
import subprocess
assert subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_link.py']) == 0